# Root-Finding Methods: Bisection, False Position, Newton, and Fixed-Point Iteration

Four independent root-finding problems, each solved with two different methods and compared by
convergence behavior (iterations needed vs. target error, against the analytic estimate
`log2((b-a)/tol)` where applicable).

In [1]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import scipy as sp
import sympy as smp
from scipy.integrate import quad


In [ ]:

# Plot style: white background with a clearly visible grid (cosmetic only)
sns.set_style("whitegrid")
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.edgecolor"] = "black"
plt.rcParams["grid.color"] = "black"
plt.rcParams["grid.linestyle"] = "-"
plt.rcParams["grid.linewidth"] = 1
plt.rcParams["xtick.color"] = "black"
plt.rcParams["ytick.color"] = "black"
plt.rcParams["axes.labelcolor"] = "black"
plt.rcParams["axes.titlecolor"] = "black"


## 1. Bisection vs. false position for `sin(exp(-3x^2 + 2)) - 2x^3 = 0` on `[0, 2]`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Function under study
def f(x):
    return np.sin(np.exp(-3*x**2 + 2)) - 2*x**3

# General parameters
error_tol = 1e-2
max_iter = 1000

# Interval [a, b]
a, b = 0, 2

# Bisection method
def bisection(a, b, f, error_tol):
    valid_bracket = 0
    error = 1
    iterations = 0
    x_prev = 0

    # Validate the bracket
    if f(a) * f(b) < 0:
        valid_bracket = 1
    elif f(a) * f(b) == 0:
        if f(a) == 0:
            return a, 0, 0
        if f(b) == 0:
            return b, 0, 0
        return None, None, None

    estimated_iterations = int(np.log2((b - a) / error_tol))

    while valid_bracket == 1 and error > error_tol and iterations < max_iter:
        x_new = (a + b) / 2

        if f(a) * f(x_new) < 0:
            b = x_new
        else:
            a = x_new

        if x_new != 0:
            error = abs((x_new - x_prev) / x_new)
        else:
            error = 0

        x_prev = x_new
        iterations += 1

        if f(x_new) == 0:
            break

    return x_new, error, iterations

# False position method
def false_position(a, b, f, error_tol):
    x_0 = 0
    error_act = 1
    iterations = 0

    if f(a) * f(b) > 0:
        raise ValueError("The function does not change sign over the given interval.")

    while error_act > error_tol and iterations < max_iter:
        fa, fb = f(a), f(b)
        x_1 = a - ((b - a) * fa) / (fb - fa)
        fx = f(x_1)

        if fx == 0:
            break
        elif fa * fx < 0:
            b = x_1
        else:
            a = x_1

        if x_1 != 0:
            error_act = abs((x_1 - x_0) / x_1)
        else:
            error_act = 0

        x_0 = x_1
        iterations += 1

    return x_0, error_act, iterations

# Roots and results for each method
methods = {
    'Bisection': bisection,
    'False position': false_position
}

for method, function in methods.items():
    try:
        root, error, iterations = function(a, b, f, error_tol)
        iter_est = int(np.log2((b - a) / error_tol))
        print(f"\nResults for the {method} method:")
        print(f"Proposed root: {root}")
        print(f"Function evaluated at the root: {f(root)}")
        print(f"Error reached: {error}")
        print(f"Number of iterations: {iterations}")
        print(f"Estimated number of iterations: {iter_est}")
    except ValueError as e:
        print(f"\n{method}: {e}")

# Plot of the function and the roots
x_vals = np.linspace(a, b, 1000)
y_vals = f(x_vals)

plt.figure(figsize=(14, 8))
plt.plot(x_vals, y_vals, label='f(x)', color='black')
plt.axhline(0, color='gray', linestyle='--')

for method, function in methods.items():
    try:
        root, _, _ = function(a, b, f, error_tol)
        plt.scatter(root, 0, label=f'{method} root: {root:.6f}', s=100)
    except ValueError:
        pass

plt.xlabel('x')
plt.ylabel('f(x)')
plt.legend()
plt.grid(True)
plt.title('Approximate Roots via Bisection and False Position')
plt.savefig("bisection_vs_false_position_survey.png", dpi=150, bbox_inches="tight")
plt.show()

# Iterations vs. target error for each method
test_tolerances = [10**-2, 10**-4, 10**-6, 10**-8]

for method, function in methods.items():
    iteration_counts = []
    for tol in test_tolerances:
        try:
            trial_counts = []
            for _ in range(8):
                _, _, it = function(a, b, f, tol)
                trial_counts.append(it)
            iteration_counts.append(np.mean(trial_counts))
        except ValueError:
            iteration_counts.append(None)

    plt.figure(figsize=(12, 8))
    plt.plot(test_tolerances, iteration_counts, marker='o', label=method)
    plt.xscale('log')
    plt.xlabel('Target error')
    plt.ylabel('Number of Iterations')
    plt.xticks(test_tolerances, labels=[f'$10^{{-{int(np.log10(e))}}}$' for e in test_tolerances])
    plt.grid(True)
    plt.legend()
    plt.title(f'Number of Iterations vs. Target Error ({method})')
    plt.show()


## 2. Newton vs. false position for a highly oscillatory function on `[-1, 0]`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Function f(x)
def f(x):
    return np.sin(7 * x) * np.exp(-x / 9) - 4 * np.cos(5 * x) * np.exp(x / 5) - 12 * np.sin(7 * x) * np.exp(-x / 7) + x**3 / 21 - 5

# Approximate derivative of f(x)
def df(x, dx=1e-6):
    return (f(x + dx) - f(x)) / dx

# Newton's method
def newton_method(x0, tol=1e-6, max_iter=100):
    error = 1
    iteration = 0
    iterations = []
    errors = []

    while error > tol and iteration < max_iter:
        x_new = x0 - f(x0) / df(x0)
        error = abs((x_new - x0) / x_new)
        iterations.append(iteration)
        errors.append(error)
        iteration += 1
        x0 = x_new

    return x0, errors, iterations

# False position method
def false_position(a, b, tol=1e-6, max_iter=100):
    x0 = 0
    error = 1
    iteration = 0
    iterations = []
    errors = []

    if f(a) * f(b) > 0:
        return None, [], []

    while error > tol and iteration < max_iter:
        fa, fb = f(a), f(b)
        x1 = a - ((b - a) * fa) / (fb - fa)
        fx = f(x1)

        error = abs((x1 - x0) / x1)
        iterations.append(iteration)
        errors.append(error)

        if fx == 0:
            break
        elif fa * fx < 0:
            b = x1
        else:
            a = x1

        x0 = x1
        iteration += 1

    return x0, errors, iterations

# General parameters
a, b = -1, 0  # Interval for false position
x0 = (a + b) / 2  # Initial guess for Newton, inside the interval

# Iterations for different target errors
test_tolerances = [10**-2, 10**-4, 10**-6, 10**-8]

# Newton's method results
print("\nNewton's Method Results:")
for tol in test_tolerances:
    root, errors, iterations = newton_method(x0, tol)
    print(f"\nTarget error: {tol}")
    print(f"Proposed root: {root:.6f}")
    print(f"Function evaluated at the root: {f(root):.6e}")
    print(f"Error reached: {errors[-1]:.6e}")
    print(f"Number of iterations: {len(errors)}")

# False position results
print("\nFalse Position Results:")
for tol in test_tolerances:
    root, errors, iterations = false_position(a, b, tol)
    if root is not None:
        print(f"\nTarget error: {tol}")
        print(f"Proposed root: {root:.6f}")
        print(f"Function evaluated at the root: {f(root):.6e}")
        print(f"Error reached: {errors[-1]:.6e}")
        print(f"Number of iterations: {len(errors)}")
    else:
        print(f"\nTarget error: {tol}")
        print("No root found in the given interval.")

# Iterations vs. target error plot
plt.figure(figsize=(12, 8))

newton_iterations = [len(newton_method(x0, tol)[1]) for tol in test_tolerances]
false_position_iterations = [len(false_position(a, b, tol)[1]) if false_position(a, b, tol)[0] is not None else 0 for tol in test_tolerances]

plt.plot(test_tolerances, newton_iterations, marker='o', label="Newton's method", color='blue')
plt.plot(test_tolerances, false_position_iterations, marker='o', label='False position', color='green')
plt.xscale('log')
plt.xlabel('Target error')
plt.ylabel('Number of Iterations')
plt.grid(True)
plt.legend()
plt.title('Number of Iterations vs. Target Error (Newton and False Position)')
plt.show()

## 3. Fixed-point iteration vs. Newton's method for `(e^-x - 0.5) sin(x) - x + 5 = 0`

### Fixed-point iteration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Function f(x) and iteration function g(x)
def f(x):
    return (np.exp(-x) - 0.5) * np.sin(x) - x + 5

def g(x):
    return f(x) + x

# Approximate derivative of g(x)
def dg(x, dx=1e-6):
    return (g(x + dx) - g(x)) / dx

# Fixed-point iteration
def fixed_point_method():
    error = 1
    tol = 1e-6
    attempts = 0
    max_attempts = 2
    iteration = 0
    max_iter = 100

    # Propose an initial value x0
    valid = False
    while not valid and error > tol and attempts < max_attempts:
        try:
            x0 = float(input("Propose an x0: "))
            convergence_criterion = abs(dg(x0))
            print(f"Convergence criterion |g'(x0)|: {convergence_criterion:.6f}")
            if convergence_criterion < 1:
                valid = True
            else:
                print("The derivative of g(x0) is >= 1. Try another value.")
            if f(x0) == 0:
                error = 0
        except ValueError:
            print("Invalid input. Enter a numeric value.")
        attempts += 1

    if not valid:
        print("No suitable initial value was found.")
        return None

    # Fixed-point iteration loop
    x_prev = x0
    iterations = []
    errors = []

    while valid and error > tol and iteration < max_iter:
        x_new = g(x_prev)
        error = abs((x_new - x_prev) / x_new)
        iterations.append(iteration)
        errors.append(error)

        if abs(dg(x_new)) >= 1:
            valid = False

        iteration += 1
        x_prev = x_new

    # Results
    print("\nSolution:", x_prev)
    print("Function evaluated at the solution:", f(x_prev))
    print("Error reached:", error)
    print("Number of iterations:", iteration)

    # Plot of the function and the root
    x_vals = np.linspace(-1, 2, 1000)
    y_vals = f(x_vals)

    plt.figure(figsize=(12, 8))
    plt.plot(x_vals, y_vals, label='f(x)', color='black')
    plt.axhline(0, color='gray', linestyle='--')
    plt.scatter(x_prev, 0, color='red', s=100, label=f'Root: {x_prev:.6f}')
    plt.xlabel('x')
    plt.ylabel('f(x)')
    plt.legend()
    plt.grid(True)
    plt.title('Approximate Root via Fixed-Point Iteration')
    plt.show()

    # Iterations for different target errors
    test_tolerances = [10**-12, 10**-10, 10**-8, 10**-6, 10**-4, 10**-2]
    test_iteration_counts = []

    for tol in test_tolerances:
        x_prev = x0
        error = 1
        iteration = 0
        while error > tol and iteration < max_iter:
            x_new = g(x_prev)
            error = abs((x_new - x_prev) / x_new)
            x_prev = x_new
            iteration += 1
        test_iteration_counts.append(iteration)

    plt.figure(figsize=(12, 8))
    plt.plot(test_tolerances, test_iteration_counts, marker='o', color='green', label='Iterations vs. Target Error')
    plt.xscale('log')
    plt.xlabel('Target error')
    plt.ylabel('Number of Iterations')
    plt.grid(True)
    plt.legend()
    plt.title('Number of Iterations vs. Target Error (Fixed-Point Iteration)')
    plt.show()

fixed_point_method()


### Newton's method

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Function f(x)
def f(x):
    return (np.exp(-x) - 0.5) * np.sin(x) - x + 5

# Approximate derivative of f(x)
def df(x, dx=1e-6):
    return (f(x + dx) - f(x)) / dx

# Approximate derivative of g(x)
def dg(x, dx=1e-6):
    g = lambda x: f(x) + x
    return (g(x + dx) - g(x)) / dx

# Newton's method
def newton_method():
    error = 1
    tol = 1e-6
    attempts = 0
    max_attempts = 2
    iteration = 0
    max_iter = 100

    # Propose an initial value x0
    valid = False
    while not valid and error > tol and attempts < max_attempts:
        try:
            x0 = float(input("Propose an x0: "))
            g_derivative = abs(dg(x0))
            print(f"Convergence criterion |g'(x0)|: {g_derivative:.6f}")
            if abs(df(x0)) > 0:
                valid = True
            else:
                print("The derivative of f(x0) is zero or too small. Try another value.")
            if f(x0) == 0:
                error = 0
        except ValueError:
            print("Invalid input. Enter a numeric value.")
        attempts += 1

    if not valid:
        print("No suitable initial value was found.")
        return None

    # Newton's method iteration
    x_prev = x0
    iterations = []
    errors = []

    while valid and error > tol and iteration < max_iter:
        x_new = x_prev - f(x_prev) / df(x_prev)
        error = abs((x_new - x_prev) / x_new)
        iterations.append(iteration)
        errors.append(error)

        iteration += 1
        x_prev = x_new

    # Results
    print("\nSolution:", x_prev)
    print("Function evaluated at the solution:", f(x_prev))
    print("Error reached:", error)
    print("Number of iterations:", iteration)
    print(f"Value of g'(x) at the solution: {dg(x_prev):.6f}")

    # Plot of the function and the root
    x_vals = np.linspace(-1, 2, 1000)
    y_vals = f(x_vals)

    plt.figure(figsize=(12, 8))
    plt.plot(x_vals, y_vals, label='f(x)', color='black')
    plt.axhline(0, color='gray', linestyle='--')
    plt.scatter(x_prev, 0, color='red', s=100, label=f'Root: {x_prev:.6f}')
    plt.xlabel('x')
    plt.ylabel('f(x)')
    plt.legend()
    plt.grid(True)
    plt.title("Approximate Root via Newton's Method")
    plt.show()

    # Iterations for different target errors
    test_tolerances = [10**-12, 10**-10, 10**-8, 10**-6, 10**-4, 10**-2]
    test_iteration_counts = []

    for tol in test_tolerances:
        x_prev = x0
        error = 1
        iteration = 0
        while error > tol and iteration < max_iter:
            x_new = x_prev - f(x_prev) / df(x_prev)
            error = abs((x_new - x_prev) / x_new)
            x_prev = x_new
            iteration += 1
        test_iteration_counts.append(iteration)

    # Iterations vs. target error plot
    plt.figure(figsize=(12, 8))
    plt.plot(test_tolerances, test_iteration_counts, marker='o', color='green', label='Iterations vs. Target Error')
    plt.xscale('log')
    plt.xlabel('Target error')
    plt.ylabel('Number of Iterations')
    plt.grid(True)
    plt.legend()
    plt.title("Number of Iterations vs. Target Error (Newton's Method)")
    plt.show()

# Run Newton's method
newton_method()


## 4. Newton-Raphson for a 2D nonlinear system

In [ ]:
import numpy as np

# Functions u(x, y) and v(x, y)
def u(x, y):
    return np.sin(np.exp((x**2)- (y**2))) - 3*y

def v(x, y):
    return np.cos(np.cos(x*y**3)) - 3*x**2*y

# General parameters
delta = 1e-4
error_tol = 1e-6
max_attempts = 2
max_iterations = 1000

# Approximate derivatives
def du_dx(x, y):
    return (u(x + delta, y) - u(x, y)) / delta

def du_dy(x, y):
    return (u(x, y + delta) - u(x, y)) / delta

def dv_dx(x, y):
    return (v(x + delta, y) - v(x, y)) / delta

def dv_dy(x, y):
    return (v(x, y + delta) - v(x, y)) / delta

# Jacobian determinant
def jacobian(x, y):
    return du_dx(x, y) * dv_dy(x, y) - du_dy(x, y) * dv_dx(x, y)

# Flags and errors
valid_bracket = 0
error_x = 1
error_y = 1
converged = 0
attempts = 0

# First loop: propose initial values
while valid_bracket == 0 and converged == 0 and attempts < max_attempts:
    try:
        x0 = float(input("Give me the initial value for x: "))
        y0 = float(input("Give me the initial value for y: "))
        
        if jacobian(x0, y0) != 0:
            valid_bracket = 1
        
        if u(x0, y0) == 0 and v(x0, y0) == 0:
            converged = 1

        attempts += 1

    except ValueError:
        print("Invalid input. Please enter numeric values.")

# Second loop: iterative method
x_prev, y_prev = x0, y0
iterations = 0

while valid_bracket == 1 and converged == 0 and iterations < max_iterations:
    coef_a = -u(x_prev, y_prev) + du_dx(x_prev, y_prev) * x_prev + du_dy(x_prev, y_prev) * y_prev
    coef_b = -v(x_prev, y_prev) + dv_dx(x_prev, y_prev) * x_prev + dv_dy(x_prev, y_prev) * y_prev

    # Compute x_new and y_new
    try:
        x_new = (coef_a * dv_dy(x_prev, y_prev) - coef_b * du_dy(x_prev, y_prev)) / jacobian(x_prev, y_prev)
        y_new = (coef_b * du_dx(x_prev, y_prev) - coef_a * dv_dx(x_prev, y_prev)) / jacobian(x_prev, y_prev)
    except ZeroDivisionError:
        print("Error: the Jacobian is zero, cannot continue.")
        break

    # Check whether the solution is exact
    if u(x_new, y_new) == 0 and v(x_new, y_new) == 0:
        converged = 1

    # Compute errors
    error_x = abs((x_new - x_prev) / x_new)
    error_y = abs((y_new - y_prev) / y_new)

    # Check whether the error is below the threshold
    if error_x < error_tol and error_y < error_tol:
        converged = 1

    # Check the Jacobian at the new iterate
    if jacobian(x_new, y_new) == 0:
        valid_bracket = 0

    # Update values
    x_prev = x_new
    y_prev = y_new
    iterations += 1

# Final results
print("\nSolution:")
print(f"x = {x_new:.6f}, y = {y_new:.6f}")

print("\nEvaluation at the solution:")
print(f"u(x, y) = {u(x_new, y_new):.6f}")
print(f"v(x, y) = {v(x_new, y_new):.6f}")

print("\nIndividual errors:")
print(f"Error in x: {error_x:.6e}")
print(f"Error in y: {error_y:.6e}")

print("\nIterations performed:")
print(iterations)
